# Lane E: LightGBM account ranking (Q1)

**What this notebook does:** trains a LightGBM model on the train period, uses the validation period to decide when to stop, and checks how well it ranks illicit accounts.

**Run order:** run every cell from top to bottom once (Runtime → Run all).
After that, if you change a setting in **Step 3**, rerun Step 3 and everything below it.

| Step | What it is |
|---|---|
| 1 | Imports |
| 2 | Fake data (skip it once D's data arrives) |
| 3 | Settings: the only cell you normally edit |
| 4–8 | The helper functions, one per cell: open a file, decile table, top-K and scheme-group metrics, the full report, training |
| 9 | Load the data, apply the optional subset, check one row per account per period |
| 10 | Leak check (does nothing unless it's switched on) |
| 11 | Pick the features |
| 12 | Train and score, with and without Taint |
| 13 | Save the results |
| 14 | Copy to Google Drive (optional) |

**Split used (lane B, RESULTS.md):** train Sep 1–8 · gap 9–10 · val Sep 11–12 · gap 13–14 · test Sep 15–16 · tail Sep 17–28 dropped.

## Step 1 · Imports
Loads the libraries. Uncomment the `pip` line if Colab says `lightgbm` is missing.

In [1]:
# !pip install lightgbm
import os
from types import SimpleNamespace

import lightgbm as lgb                                    # the model
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score       # the AP metric

In [2]:
# Step 2b · Connect to S3 (run once per session) and list processed/
!pip -q install boto3
import boto3
from collections import Counter
from google.colab import userdata

BUCKET = "is459-g2t8-aml-897729121624"
s3 = boto3.client(
    "s3", region_name="us-east-1",
    aws_access_key_id=userdata.get("AWS_ACCESS_KEY_ID").strip(),
    aws_secret_access_key=userdata.get("AWS_SECRET_ACCESS_KEY").strip(),
)

# show every folder under processed/ and how many files it holds, to find D's output
folders = Counter()
for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix="processed/"):
    for o in page.get("Contents", []):
        folders[o["Key"].rsplit("/", 1)[0]] += 1
for f, n in sorted(folders.items()):
    print(f"{n:5d} files  {f}/")

    1 files  processed/
    1 files  processed/dq_report/
    5 files  processed/pattern_transactions/
    1 files  processed/patterns/
    1 files  processed/rejected/
    1 files  processed/reports/
    1 files  processed/transactions/transaction_date=2022-09-01/
    1 files  processed/transactions/transaction_date=2022-09-02/
    1 files  processed/transactions/transaction_date=2022-09-03/
    1 files  processed/transactions/transaction_date=2022-09-04/
    1 files  processed/transactions/transaction_date=2022-09-05/
    1 files  processed/transactions/transaction_date=2022-09-06/
    1 files  processed/transactions/transaction_date=2022-09-07/
    1 files  processed/transactions/transaction_date=2022-09-08/
    1 files  processed/transactions/transaction_date=2022-09-09/
    1 files  processed/transactions/transaction_date=2022-09-10/
    1 files  processed/transactions/transaction_date=2022-09-11/
    1 files  processed/transactions/transaction_date=2022-09-12/
    1 files  proces

In [3]:
# Step 2c · Download D's features (and labels, if separate) from S3
FEATURES_PREFIX = "results/rule-features/q1/features/"    # D's folder, copied from the list above
LABELS_PREFIX   = None                       # e.g. "processed/labels/" if labels are a separate folder

def download_prefix(prefix, local_dir):
    n = 0
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=prefix):
        for o in page.get("Contents", []):
            k = o["Key"]
            if k.endswith("/") or k.endswith("$folder$") or "/_" in k:   # skip folder markers and _SUCCESS
                continue
            rel = os.path.relpath(k, prefix)
            if rel == ".":                                                # prefix was a single file
                rel = os.path.basename(k)
            dest = os.path.join(local_dir, rel)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            s3.download_file(BUCKET, k, dest)
            n += 1
    print(f"downloaded {n} files from s3://{BUCKET}/{prefix} -> {local_dir}/")

download_prefix(FEATURES_PREFIX, "real/features")
if LABELS_PREFIX:
    download_prefix(LABELS_PREFIX, "real/labels")

# quick look: size, column names and types, first rows
files = [f for _, _, fs in os.walk("real/features") for f in fs]
peek = pd.read_csv(os.path.join("real/features", files[0])) if files[0].endswith(".csv") else pd.read_parquet("real/features")
print(peek.shape)
print(peek.dtypes)
peek.head()

downloaded 24 files from s3://is459-g2t8-aml-897729121624/results/rule-features/q1/features/ -> real/features/
(4057221, 12)
acct             object
n_out             int64
d_out             int64
usd_out         float64
n_in              int64
d_in              int64
usd_in          float64
fan_ratio       float64
flow_match      float64
chain_score     float64
taint           float64
period         category
dtype: object


,acct,n_out,d_out,usd_out,n_in,d_in,usd_in,fan_ratio,flow_match,chain_score,taint,period
0,000|800071D50,43,13,1.230241e+05,2,1,2.930000e+00,0.076923,0.002382,95.705521,0.0,test
1,000|800074310,44,10,1.398327e+05,8,2,7.225922e+04,0.200000,51.675467,94.031731,1.0,test
2,000|80010F380,48,13,2.804442e+05,9,3,6.703678e+05,0.230769,41.834378,99.455171,1.0,test
3,000|800114BB0,39,9,3.236883e+05,9,2,1.354250e+08,0.222222,0.239017,85.064029,0.0,test
4,000|8001946E0,6,3,1.047690e+06,0,0,0.000000e+00,0.000000,0.000000,0.000000,0.0,test


In [4]:
# Step 2d · Build the account labels locally in Colab (reads from S3, changes nothing in AWS)
# Needs Step 2b (the S3 connection) and Step 2c (download_prefix, and D's features in real/features).
!pip -q install duckdb
import duckdb

# 1. Download the cleaned transactions (about 1.2 GB) and the parsed Patterns.txt (2.6 MB)
download_prefix("processed/transactions/", "real/transactions")
download_prefix("processed/patterns/", "real/patterns")

# 2. Lane B's split dates (q1b-split/out/RESULTS.md)
TRAIN_END, GAP1_END, VAL_END, GAP2_END, TEST_END = "2022-09-08", "2022-09-10", "2022-09-12", "2022-09-14", "2022-09-16"

con = duckdb.connect()

# 3. Every transaction: sender, receiver, label, and which period it falls in
con.execute(f"""
CREATE OR REPLACE VIEW t AS
SELECT CAST(from_bank AS VARCHAR) || '|' || CAST(from_account AS VARCHAR) AS src,   -- same key format as D's acct
       CAST(to_bank   AS VARCHAR) || '|' || CAST(to_account   AS VARCHAR) AS dst,
       txn_ts,
       CAST(is_laundering AS INTEGER) AS y,
       CASE WHEN CAST(txn_ts AS DATE) <= DATE '{TRAIN_END}' THEN 'train'
            WHEN CAST(txn_ts AS DATE) <= DATE '{GAP1_END}'  THEN 'gap1'
            WHEN CAST(txn_ts AS DATE) <= DATE '{VAL_END}'   THEN 'val'
            WHEN CAST(txn_ts AS DATE) <= DATE '{GAP2_END}'  THEN 'gap2'
            WHEN CAST(txn_ts AS DATE) <= DATE '{TEST_END}'  THEN 'test'
            ELSE 'tail' END AS period
FROM read_parquet('real/transactions/**/*.parquet', hive_partitioning = true)
""")

# 4. Patterns.txt: when each scheme began, and the earliest scheme start for every account in one
PCOLS = ["attempt_id", "pattern_type", "pattern_detail", "ts_text", "from_bank", "from_account", "to_bank",
         "to_account", "amount_received", "receiving_currency", "amount_paid", "payment_currency",
         "payment_format", "is_laundering"]
pfile = [os.path.join(d, f) for d, _, fs in os.walk("real/patterns") for f in fs if f.endswith(".csv")][0]
has_header = "attempt_id" in open(pfile).readline()
src = (f"read_csv('{pfile}', all_varchar = true, header = true)" if has_header else
       f"read_csv('{pfile}', all_varchar = true, header = false, names = {PCOLS})")  # all_varchar keeps bank IDs like '000'
con.execute(f"""
CREATE OR REPLACE TABLE acct_scheme AS
WITH p AS (
  SELECT attempt_id, from_bank || '|' || from_account AS src, to_bank || '|' || to_account AS dst,
         strptime(ts_text, '%Y/%m/%d %H:%M') AS ts
  FROM {src}),
s AS (SELECT attempt_id, min(ts) AS attempt_start FROM p GROUP BY 1)
SELECT acct, min(attempt_start) AS first_attempt_start
FROM (SELECT p.src AS acct, s.attempt_start FROM p JOIN s USING (attempt_id)
      UNION ALL
      SELECT p.dst, s.attempt_start FROM p JOIN s USING (attempt_id))
GROUP BY 1
""")

# 5. One row per account per period: counts, is_illicit, first_seen_period, pos_group
os.makedirs("real/labels", exist_ok=True)
con.execute(f"""
COPY (
  WITH e AS (
    SELECT src AS acct, period, txn_ts, 1 AS sent, 0 AS recv, y AS laund_sent, 0 AS laund_recv FROM t
    UNION ALL
    SELECT dst,        period, txn_ts, 0,         1,         0,             y          FROM t),
  ap AS (
    SELECT acct, period, sum(sent) AS txns_sent, sum(recv) AS txns_recv,
           sum(laund_sent) AS laund_sent, sum(laund_recv) AS laund_recv, min(txn_ts) AS first_ts
    FROM e GROUP BY 1, 2),
  fs AS (SELECT acct, arg_min(period, first_ts) AS first_seen_period FROM ap GROUP BY 1)
  SELECT ap.acct, ap.period, ap.txns_sent, ap.txns_recv, ap.laund_sent, ap.laund_recv,
         CASE WHEN ap.laund_sent + ap.laund_recv > 0 THEN 1 ELSE 0 END AS is_illicit,  -- sent OR received laundering
         fs.first_seen_period,
         CASE WHEN ap.laund_sent + ap.laund_recv = 0 THEN ''
              WHEN a.first_attempt_start IS NULL THEN 'none'                           -- laundering, no scheme in Patterns.txt
              WHEN a.first_attempt_start > TIMESTAMP '{TRAIN_END} 23:59:59' THEN 'new' -- all its schemes began after train
              ELSE 'old' END AS pos_group                                              -- a scheme began during train
  FROM ap JOIN fs USING (acct) LEFT JOIN acct_scheme a USING (acct)
  WHERE ap.period IN ('train', 'val', 'test')
) TO 'real/labels/labels.parquet' (FORMAT PARQUET)
""")

# 6. Checks: counts per period, and whether the keys match D's features
lab = pd.read_parquet("real/labels/labels.parquet")
print(f"labels: {len(lab):,} rows")
print(lab.groupby("period").agg(accounts=("acct", "size"), illicit=("is_illicit", "sum"),
                                illicit_rate=("is_illicit", "mean"),
                                new=("pos_group", lambda g: (g == "new").sum()),
                                old=("pos_group", lambda g: (g == "old").sum()),
                                none=("pos_group", lambda g: (g == "none").sum())).to_string())

feat_keys = pd.read_parquet("real/features", columns=["acct", "period"])
feat_keys["period"] = feat_keys["period"].astype(str)
m = feat_keys.merge(lab[["acct", "period"]], on=["acct", "period"], how="left", indicator=True)
print(f"\nD's feature rows with a label: {(m._merge == 'both').mean():.2%} of {len(m):,}")
print("(should be close to 100%; if it is near 0%, the account key formats differ)")

downloaded 28 files from s3://is459-g2t8-aml-897729121624/processed/transactions/ -> real/transactions/
downloaded 1 files from s3://is459-g2t8-aml-897729121624/processed/patterns/ -> real/patterns/


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

labels: 4,452,368 rows
        accounts  illicit  illicit_rate   new   old  none
period                                                   
test     1448805     6890      0.004756  3383   532  2975
train    2063291    18130      0.008787    78  8254  9798
val       940272     6095      0.006482  2064  1728  2303

D's feature rows with a label: 100.00% of 4,057,221
(should be close to 100%; if it is near 0%, the account key formats differ)


## Step 3 · Settings
Everything you might change is here. After changing anything, **rerun this cell and every cell below it.**

- `features`: D's feature file. For fake data: `fake/features.csv`.
- `labels`: B's label file, only if the labels are **not** already in the features file.
- Column names: change these if D or B use different names.
- `taint_col`: the Taint column, so the model is run with and without it. `None` skips that.
- `group_col`: the column that says what kind of positive an account is (`new` / `old` / `none`). Used for the "new schemes only" numbers. Set to `None` if the file has no such column.
- `subset_col`: `None` = all accounts (the main run). A 0/1 column name, e.g. `"is_rule_flagged"`, keeps only rows where it is 1, for the "rule-flagged accounts only" comparison.
- `k_list`: the K values for precision and recall at top K.
- `drop_cols`: number columns that must **not** be used as features. `is_rule_flagged` is there by default: it only marks the subset, and D's rule scores already carry the rule information.
- `score_test`: keep `False` until the final run after the meetup. Test is scored **once**.
- `shuffle_labels`: `True` runs the leak check.

In [5]:
a = SimpleNamespace(
    features="real/features",              # D's features: results/rule-features/q1/features/ (Step 2c)
    labels="real/labels/labels.parquet",   # account labels built in Step 2d
    id_col="acct",                         # D's account column (bank|account); the labels use the same name
    period_col="period",                   # train / val / test, from D's folder names
    label_col="is_illicit",                # from the labels: 1 = sent or received laundering in that period
    first_seen_col="first_seen_period",    # from the labels
    taint_col="taint",                     # D's Rule C
    group_col="pos_group",                 # from the labels: new / old / none
    subset_col=None,                       # None = all accounts (main run)
    pct_list=(0.10, 0.25, 0.30, 0.50), # top-X% cut-offs: share of accounts checked
    # safety net: label counts must never become features, in case they get merged in
    drop_cols="txns_sent,txns_recv,laund_sent,laund_recv",
    score_test=False,                      # keep False until the final run
    transactions=None,                     # only for the final per-transaction score
    shuffle_labels=False,
    weight=True,
    lr=0.05,
    num_leaves=31,
    min_leaf=50,
    rounds=2000,
    early_stop=100,
    seed=42,
    out_dir="out",
)
os.makedirs(a.out_dir, exist_ok=True)

## Step 4 · Helper: `read_any`
Opens a file as a table. Parquet if the name ends in `.parquet` / `.pq` (or is a folder), otherwise CSV.

In [6]:
def read_any(path):
    if path.endswith((".parquet", ".pq")) or os.path.isdir(path):
        return pd.read_parquet(path)
    return pd.read_csv(path)

## Step 5 · Helper: `decile_table`
Sorts accounts from highest score to lowest, cuts them into 10 equal groups (decile 1 = top 10%), and for each group gives:

- `positives`: illicit accounts in the group
- `rate`: positives ÷ group size
- `lift`: rate ÷ overall rate (above 1 = richer in illicit accounts than average)
- `cum_recall`: share of all illicit accounts found if you check down to this group

Equal scores are ordered randomly so they don't all land in one group.

In [7]:
def decile_table(y, s, seed):
    d = pd.DataFrame({"y": np.asarray(y), "s": np.asarray(s)})
    d["tie"] = np.random.default_rng(seed).random(len(d))                        # random tie-breaker
    d = d.sort_values(["s", "tie"], ascending=[False, True]).reset_index(drop=True)
    d["decile"] = (np.arange(len(d)) * 10 // len(d)) + 1                          # 1..10
    base = d.y.mean()                                                             # overall positive rate
    t = d.groupby("decile").agg(accounts=("y", "size"), positives=("y", "sum"))
    t["rate"] = t.positives / t.accounts
    t["lift"] = t.rate / base if base > 0 else np.nan
    t["cum_recall"] = t.positives.cumsum() / max(d.y.sum(), 1)
    return t

## Step 6 · Helper: `rank_metrics`
Two extra views of the same ranked list.

**Top K.** An investigator can only check so many accounts. For each K in `k_list`:
- `precision_at_K`: of the top K accounts, what share are illicit
- `recall_at_K`: of all illicit accounts, what share are inside the top K

**By kind of positive** (needs `group_col`). Illicit accounts are split into:
- `new`: all of its schemes began after the training period ended. This is the honest group: the model cannot have seen any part of these schemes.
- `old`: at least one of its schemes began during training (it straddles the split)
- `none`: laundering with no scheme in Patterns.txt

For each group it gives the count, the recall at each K (share of that group inside the overall top K), and the AP of "all normal accounts + only that group's positives". The `new` row is the "schemes kept together" number the prof asked about, without moving any rows between periods.

In [8]:
def rank_metrics(y, s, seed, ks, groups=None, pcts=()):
    y, s = np.asarray(y), np.asarray(s)
    d = pd.DataFrame({"y": y, "s": s, "g": "" if groups is None else np.asarray(groups).astype(str)})
    d["tie"] = np.random.default_rng(seed).random(len(d))
    d = d.sort_values(["s", "tie"], ascending=[False, True])
    d["rank"] = np.arange(1, len(d) + 1)                                          # 1 = highest score
    total = max(int(y.sum()), 1)
    ks = [k for k in ks if k <= len(d)]

    topk = pd.DataFrame({"K": ks})
    topk["positives_in_top_K"] = [int(d.y[d["rank"] <= k].sum()) for k in ks]
    topk["precision_at_K"] = topk.positives_in_top_K / topk.K
    topk["recall_at_K"] = topk.positives_in_top_K / total
    topk = topk.set_index("K")

    base = y.mean()
    rows = []
    for p in pcts:                                                                # top X% of accounts
        k = max(1, int(round(p * len(d))))
        hit = int(d.y[d["rank"] <= k].sum())
        rows.append({"top": f"{p:.0%}", "accounts": k, "positives": hit, "precision": hit / k,
                     "recall": hit / total, "lift": (hit / k) / base if base > 0 else np.nan})
    toppct = pd.DataFrame(rows).set_index("top") if rows else None

    by_group = None
    if groups is not None:
        rows = []
        for g in sorted(set(d.g[d.y == 1])):
            pg = (d.y == 1) & (d.g == g)
            m = (d.y == 0) | pg                                                   # negatives + this group only
            ap_g = average_precision_score(d.y[m], d.s[m]) if pg.any() else np.nan
            row = {"group": g, "positives": int(pg.sum()), "ap": ap_g,
                   "x_random": ap_g / (pg.sum() / m.sum()) if pg.any() else np.nan}    # vs this group's own baseline
            for k in ks:
                row[f"recall_at_{k}"] = float((d["rank"][pg] <= k).mean())
            for p in pcts:
                row[f"recall_top_{p:.0%}"] = float((d["rank"][pg] <= max(1, int(round(p * len(d))))).mean())
            rows.append(row)
        by_group = pd.DataFrame(rows).set_index("group")
    return topk, toppct, by_group

## Step 7 · Helper: `report`
Works out and prints the metrics for one set of scores (`y` = true labels, `s` = model scores):

- **positive rate**: share illicit, which is also what a random score gets as AP
- **AP**: how early the illicit accounts appear in the ranked list (0 to 1)
- **× random**: AP ÷ positive rate
- **random score AP**: AP of a truly random score, a sanity check that it sits near the positive rate
- **first seen**: AP only on accounts first seen in this period
- **top K** and **by kind of positive**, from Step 6
- the **decile table** from Step 5

Returns the numbers as a dict so they can be saved.

In [9]:
def report(name, y, s, seed, ks=(100, 500, 1000), first_seen=None, period=None, groups=None, unit="accounts", pcts=()):
    y = np.asarray(y)
    n, pos = len(y), int(y.sum())
    print(f"\n=== {name} ===")
    if pos == 0:
        print(f"  {n:,} {unit}, 0 positives: AP undefined")
        return {}

    rate = pos / n                                                                 # random baseline
    ap = average_precision_score(y, s)
    ap_rand = average_precision_score(y, np.random.default_rng(seed).random(n))    # sanity check
    print(f"  {unit} {n:,} | positives {pos:,} | positive rate {rate:.4%}")
    print(f"  AP {ap:.4f}  = {ap / rate:.1f}x random   (random score AP: {ap_rand:.4f})")
    out = {"name": name, "rows": n, "positives": pos, "pos_rate": rate, "ap": ap, "x_random": ap / rate}

    if first_seen is not None and period is not None:                              # accounts new in this period
        m = np.asarray(first_seen) == period
        if m.sum() and y[m].sum():
            r2 = y[m].mean()
            ap2 = average_precision_score(y[m], np.asarray(s)[m])
            print(f"  first seen in {period}: {m.sum():,} accounts, {int(y[m].sum())} positives, "
                  f"AP {ap2:.4f} = {ap2 / r2:.1f}x random")
            out.update({"ap_first_seen": ap2, "positives_first_seen": int(y[m].sum())})

    topk, toppct, by_group = rank_metrics(y, s, seed, ks, groups, pcts)            # Step 6
    if toppct is not None:
        print(f"  top X% of {unit} checked:")
        print(toppct.rename(columns={"accounts": unit}).to_string(float_format=lambda v: f"{v:.4f}"))
        for p in toppct.index:
            tag = p.rstrip("%")
            out[f"precision_top_{tag}pct"] = toppct.precision[p]
            out[f"recall_top_{tag}pct"] = toppct.recall[p]
            out[f"lift_top_{tag}pct"] = toppct.lift[p]
    print("  top K:")
    print(topk.to_string(float_format=lambda v: f"{v:.4f}"))
    for k in topk.index:
        out[f"precision_at_{k}"] = topk.precision_at_K[k]
        out[f"recall_at_{k}"] = topk.recall_at_K[k]
    if by_group is not None and len(by_group):
        print("  by kind of positive (new = schemes began after training; the honest group):")
        print(by_group.to_string(float_format=lambda v: f"{v:.4f}"))
        for g in by_group.index:
            out[f"ap_{g}"] = by_group.ap[g]
            out[f"x_random_{g}"] = by_group.x_random[g]
            out[f"positives_{g}"] = int(by_group.positives[g])

    t = decile_table(y, s, seed).rename(columns={"accounts": unit})
    print("  deciles:")
    print(t.to_string(float_format=lambda v: f"{v:.4f}"))
    out.update({"top_decile_lift": t.lift.iloc[0], "top_decile_recall": t.cum_recall.iloc[0]})
    return out

## Step 8 · Helper: `train_one`
Trains one LightGBM model:

1. Takes the **train** rows to learn from, and the **val** rows to decide when to stop.
2. **Imbalance:** `scale_pos_weight` = normal ÷ illicit in train, so each illicit account counts as much as hundreds of normal ones. Only train is weighted; val and test keep their real share.
3. **Early stopping:** adds trees one by one and checks val AP after each; stops when it hasn't improved for `early_stop` trees, and keeps the best point.
4. Prints the **feature importance**: the share of the learning that came from each feature.

In [10]:
def train_one(df, feats, a, tag):
    tr = df[df[a.period_col] == "train"]
    va = df[df[a.period_col] == "val"]
    ytr = tr[a.label_col].to_numpy()
    yva = va[a.label_col].to_numpy()
    pos, neg = ytr.sum(), len(ytr) - ytr.sum()

    params = {
        "objective": "binary",                     # illicit or not
        "metric": "average_precision",             # what early stopping watches
        "learning_rate": a.lr, "num_leaves": a.num_leaves, "min_data_in_leaf": a.min_leaf,
        "feature_fraction": 0.8,                   # each tree sees 80% of the features
        "bagging_fraction": 0.8, "bagging_freq": 1,  # each tree sees 80% of the rows
        "scale_pos_weight": (neg / pos) if (a.weight and pos) else 1.0,   # imbalance fix, TRAIN only
        "seed": a.seed, "deterministic": True, "verbose": -1,
    }
    dtr = lgb.Dataset(tr[feats], ytr)
    dva = lgb.Dataset(va[feats], yva, reference=dtr)
    model = lgb.train(params, dtr, num_boost_round=a.rounds, valid_sets=[dva], valid_names=["val"],
                      callbacks=[lgb.early_stopping(a.early_stop, verbose=False)])

    print(f"\n[{tag}] features: {feats}")
    print(f"[{tag}] train rows {len(tr):,} ({int(pos)} positives) | "
          f"scale_pos_weight {params['scale_pos_weight']:.1f} | best round {model.best_iteration}")
    imp = pd.Series(model.feature_importance("gain"), index=feats).sort_values(ascending=False)
    print(f"[{tag}] feature importance (gain, share):")
    print((imp / imp.sum()).round(3).to_string())
    return model

## Step 9 · Load data
1. Reads the feature file.
2. If a separate labels file is given, joins it on account + period. Rows with no label count as 0, and the count is printed.
3. If `subset_col` is set, keeps only the rows where that column is 1 (e.g. rule-flagged accounts only).
4. Checks there is **one row per account per period** (stops with an error if not).
5. Prints rows, positives and positive rate per period, which should match the numbers B posts.

In [11]:
df = read_any(a.features)
df[a.period_col] = df[a.period_col].astype(str)                 # D's period comes in as a category

if a.labels:
    lab = read_any(a.labels).rename(columns={"account_key": a.id_col})
    lab[a.period_col] = lab[a.period_col].astype(str)
    keep = [c for c in [a.id_col, a.period_col, a.label_col, a.first_seen_col, a.group_col] if c and c in lab.columns]
    df = df.drop(columns=[c for c in keep[2:] if c in df.columns]).merge(lab[keep], on=[a.id_col, a.period_col], how="left")
    missing = df[a.label_col].isna().sum()
    print(f"labels matched for {len(df) - missing:,} of {len(df):,} rows")
    if missing:
        print(f"!! {missing:,} feature rows have no label; treated as 0")
    df[a.label_col] = df[a.label_col].fillna(0)
    if a.group_col in df.columns:
        df[a.group_col] = df[a.group_col].fillna("")
df[a.label_col] = df[a.label_col].astype(int)

labels matched for 4,057,221 of 4,057,221 rows


## Step 10 · Leak check (only when `shuffle_labels=True`)
Shuffles the labels inside each period, keeping each period's number of positives. The labels then mean nothing, so **every AP should come out near 1× random**. If one doesn't, the code is passing the answer to the model somehow.

It can't catch Taint built from labels inside D's job. The with/without Taint comparison in Step 12 is for that.

Does nothing when `shuffle_labels=False`.

In [12]:
if a.shuffle_labels:
    rng = np.random.default_rng(a.seed)
    for per, idx in df.groupby(a.period_col).groups.items():
        df.loc[idx, a.label_col] = rng.permutation(df.loc[idx, a.label_col].to_numpy())
    print("!! LEAK CHECK: labels shuffled in every period; all APs should be near 1x random")
else:
    print("leak check off")

leak check off


## Step 11 · Pick the features
Every **number** column becomes a feature, except the account key, period, label, `first_seen_period`, `group_col`, `subset_col` and anything in `drop_cols`. So new score columns from D are picked up automatically.

Sets up two runs: **with_taint** (all features) and **no_taint** (all except `taint_col`).

In [13]:
non_feat = {a.id_col, a.period_col, a.label_col, a.first_seen_col, a.group_col, a.subset_col} | {c for c in a.drop_cols.split(",") if c}
all_feats = [c for c in df.columns if c not in non_feat and pd.api.types.is_numeric_dtype(df[c])]

runs = [("with_taint", all_feats)]
if a.taint_col:
    assert a.taint_col in all_feats, f"taint_col {a.taint_col} not found among numeric features"
    runs.append(("no_taint", [c for c in all_feats if c != a.taint_col]))

print("features:", all_feats)
print("runs:", [r[0] for r in runs])

features: ['n_out', 'd_out', 'usd_out', 'n_in', 'd_in', 'usd_in', 'fan_ratio', 'flow_match', 'chain_score', 'taint']
runs: ['with_taint', 'no_taint']


## Step 12 · Train and score
For each run (with and without Taint):

1. Train with `train_one` (Step 8).
2. Score **val**, plus **test** only if `score_test=True`.
3. Print the metrics with `report` (Step 7), per account.
4. If a transactions file is given (test only): a transaction's score = the **higher** of its sender's and receiver's scores, then the same metrics per transaction. Accounts with no score get the lowest score.

In [14]:
periods = ["val"] + (["test"] if a.score_test else [])
fs = df[a.first_seen_col] if a.first_seen_col in df.columns else None
gr = df[a.group_col] if a.group_col and a.group_col in df.columns else None
summary, scores = [], []

for tag, feats in runs:
    model = train_one(df, feats, a, tag)

    for per in periods:
        m = df[a.period_col] == per
        s = model.predict(df.loc[m, feats], num_iteration=model.best_iteration)

        # slice the optional helpers only when they exist
        fs_m = fs.loc[m] if fs is not None else None
        gr_m = gr.loc[m] if gr is not None else None

        # per account
        r = report(
            f"{tag} | {per} | per account",
            df.loc[m, a.label_col].to_numpy(), s, a.seed,
            first_seen=fs_m,
            period=per,
            groups=gr_m,
            unit="accounts",
            pcts=a.pct_list,
        )
        if r:
            summary.append({"model": tag, "period": per, "unit": "account", **r})

        scores.append(pd.DataFrame({
            a.id_col: df.loc[m, a.id_col].to_numpy(),
            "period": per,
            "model": tag,
            "score": s,
            "label": df.loc[m, a.label_col].to_numpy(),
        }))

        # per transaction (test only)
        if per == "test" and a.transactions:
            tx = read_any(a.transactions)
            sc = dict(zip(df.loc[m, a.id_col], s))
            lo = float(np.min(s))
            st = np.maximum(
                tx.from_key.map(sc).fillna(lo).to_numpy(),
                tx.to_key.map(sc).fillna(lo).to_numpy(),
            )
            r = report(
                f"{tag} | test | per transaction (max of sender, receiver)",
                tx.is_laundering, st, a.seed,
                unit="transactions",
            )
            if r:
                summary.append({"model": tag, "period": "test", "unit": "transaction", **r})


[with_taint] features: ['n_out', 'd_out', 'usd_out', 'n_in', 'd_in', 'usd_in', 'fan_ratio', 'flow_match', 'chain_score', 'taint']
[with_taint] train rows 1,668,817 (18130 positives) | scale_pos_weight 91.0 | best round 242
[with_taint] feature importance (gain, share):
fan_ratio      0.276
d_in           0.171
usd_in         0.155
usd_out        0.151
n_out          0.081
d_out          0.059
n_in           0.038
chain_score    0.032
taint          0.023
flow_match     0.014

=== with_taint | val | per account ===
  accounts 939,804 | positives 6,095 | positive rate 0.6485%
  AP 0.0436  = 6.7x random   (random score AP: 0.0064)
  first seen in val: 3,529 accounts, 78 positives, AP 0.1127 = 5.1x random
  top X% of accounts checked:
     accounts  positives  precision  recall   lift
top                                               
10%     93980       2428     0.0258  0.3984 3.9836
25%    234951       4108     0.0175  0.6740 2.6960
30%    281941       4405     0.0156  0.7227 2.4091
50%

## Step 13 · Save the results
- `out/summary.csv`: one row per model × period × unit (AP, × random, top-decile lift, …). This is what goes on the slide.
- `out/account_scores.csv`: every account's score. This is what C and F need.

Find them with the **folder icon** on the left → `out`. They are deleted when the Colab session ends, so download them or run Step 14.

In [15]:
pd.DataFrame(summary).to_csv(os.path.join(a.out_dir, "summary.csv"), index=False)
pd.concat(scores).to_csv(os.path.join(a.out_dir, "account_scores.csv"), index=False)
print(f"Saved {a.out_dir}/summary.csv and {a.out_dir}/account_scores.csv")
cols = ["model", "period", "unit", "positives", "pos_rate", "ap", "x_random", "x_random_new",
        "recall_top_10pct", "recall_top_30pct", "lift_top_30pct", "precision_at_100"]
pd.DataFrame(summary).reindex(columns=cols)

Saved out/summary.csv and out/account_scores.csv


,model,period,unit,positives,pos_rate,ap,x_random,x_random_new,recall_top_10pct,recall_top_30pct,lift_top_30pct,precision_at_100
0,with_taint,val,account,6095,0.006485,0.043636,6.728422,14.875608,0.398359,0.722724,2.409080,0.71
1,no_taint,val,account,6095,0.006485,0.046343,7.145732,15.759233,0.448400,0.712715,2.375719,0.72


## Step 14 · Save to Google Drive (optional)
Copies the `out` folder to your Drive so it survives the session. Asks for Drive permission the first time.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p "/content/drive/MyDrive/IS459/laneE"
# !cp -r out "/content/drive/MyDrive/IS459/laneE/"